# Бустинг поверх семантики (e5-large)

Идея:
1. кандидатогенерация — топ по косинусу внутри гео-пула (`K_GEO`) + немного топа без гео (`K_GLOBAL`)
2. LightGBM-классификатор скорит кандидатов, берём топ-50 по вероятности

Трейн для бустинга — пары (запрос, кандидат) по запросам из train, которые не попали в валидацию. Таргет = 1, если объявление реально выбрали. Негативы — невыбранные кандидаты того же запроса (т.е. хард-негативы: похожие и рядом).

Пара моментов, т.к. это бинарная, а не ранкер:
- косинус у разных запросов в разном масштабе, поэтому добавил относительные фичи (ранг в запросе, отставание от топ-1)
- метрика усредняется по запросам, т.е. вес позитива = 1 / кол-во правильных ответов у запроса, чтобы запросы с кучей ответов (маникюр в Мск) не перетягивали
- финально смотрю на recall@50 на валидации, logloss только для early stopping

Для запуска нужны parquet'ы и эмбеддинги e5-large на драйве

## Параметры

In [ ]:
import os
import hashlib
import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from tqdm.auto import tqdm

DATA_DIR = '/content/drive/MyDrive/avito_bootcamp/datasets'
EMB_DIR = '/content/drive/MyDrive/avito_bootcamp/emb_e5_large'
OUT_DIR = '/content/drive/MyDrive/avito_bootcamp/boosting'    # сюда модель и answer

# гео
R_KM = 40              # радиус гео-пула, подбирал в EDA-ноутбуке
COUNTRY_P50_KM = 300   # если медиана расстояния больше - считаем что это "вся россия"
MIN_PICKS = 30

# кандидаты
TOP_N = 50
N_GLOBAL = 3      # сколько мест из 50 оставляем под объявления вне гео-пула
K_GEO = 100       # 250 пробовал - не лучше, а дольше
K_GLOBAL = 50     # >= TOP_N, чтобы у каждого запроса точно было 50 кандидатов

# обучение
N_TRAIN_QUERIES = 10_000
EARLY_STOP_SHARE = 0.1
RANDOM_STATE = 42

EARTH_RADIUS_KM = 6371
SEARCH_COLS = ['search_query', 'search_location_id', 'search_is_delivery_search',
               'search_infm_params_text', 'search_category']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
print('Устройство:', DEVICE)

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except ImportError:
    pass
os.makedirs(OUT_DIR, exist_ok=True)

Устройство: cpu
Mounted at /content/drive


In [2]:
def norm(s):
    # lower + ё->е, чтобы одинаковые запросы не считались разными
    return s.fillna('').str.lower().str.replace('ё', 'е').str.strip()


def haversine_km(lat1, lon1, lat2, lon2):
    # расстояние по сфере в км, работает с массивами
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

## Данные

Тексты тут уже не нужны (эмбеддинги готовы), поэтому читаю только нужные колонки — так train в разы легче по памяти. Координаты/цена/рейтинг лежат строками, кастую в числа.

In [3]:
ITEM_COLS = ['item_id', 'item_location_id', 'item_latitude', 'item_longitude',
             'item_price', 'item_rating', 'item_rating_reviews_count', 'item_microcat_id',
             'item_is_phone_hidden', 'item_is_message_forbidden']

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'), columns=SEARCH_COLS + ITEM_COLS)
bq = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_queries.parquet'))
items = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_items.parquet'), columns=ITEM_COLS)

for df in (train, items):
    for col in ['item_latitude', 'item_longitude', 'item_price', 'item_rating', 'item_rating_reviews_count']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'train: {len(train)}, бенчмарк: {len(bq)}, корпус: {len(items)}')

train: 497673, бенчмарк: 2452, корпус: 189212


## Сплит

Запрос = группа строк train с одинаковыми `search_*`, выбранные в ней объявления — ответы.
- валидация — те же 2500 запросов, что в EDA-ноутбуке (`val_groups.parquet`), только для оценки
- под бустинг беру 10к случайных запросов из остального train, 10% из них на early stopping

In [4]:
train['query_group'] = train.groupby(SEARCH_COLS, dropna=False).ngroup()

val_groups = set(pd.read_parquet(os.path.join(EMB_DIR, 'val_groups.parquet'))['query_group'])

rest_groups = np.setdiff1d(train['query_group'].unique(), list(val_groups))
rng = np.random.default_rng(RANDOM_STATE)
boost_groups = rng.choice(rest_groups, size=N_TRAIN_QUERIES, replace=False)
n_es = int(N_TRAIN_QUERIES * EARLY_STOP_SHARE)
es_groups, fit_groups = boost_groups[:n_es], boost_groups[n_es:]

print(f'Валидация: {len(val_groups)}, обучение: {len(fit_groups)}, ранняя остановка: {len(es_groups)}')

Валидация: 2500, обучение: 9000, ранняя остановка: 1000


## Гео-фильтр

То же, что в EDA:
- центр локации объявлений = медиана координат её объявлений
- центр локации поиска = центр такой же локации объявлений, а если такой нет (регионы) — медиана координат выбранных там объявлений
- локации, где медиана расстояния до выборов > 300 км, считаем "всей Россией" и гео не фильтруем (по факту это одна 621540)
- гео-пул = все локации объявлений в радиусе `R_KM`

In [5]:
all_item_geo = (
    pd.concat([items, train])[['item_id', 'item_location_id', 'item_latitude', 'item_longitude']]
    .drop_duplicates('item_id')
)
item_loc_centers = (
    all_item_geo.groupby('item_location_id')
    .agg(lat=('item_latitude', 'median'), lon=('item_longitude', 'median'))
    .dropna()
)
train_loc_centers = (
    train.groupby('search_location_id')
    .agg(lat=('item_latitude', 'median'), lon=('item_longitude', 'median'))
)
search_loc_centers = pd.concat([item_loc_centers, train_loc_centers])
search_loc_centers = search_loc_centers[~search_loc_centers.index.duplicated(keep='first')]

# ищем локации типа "вся страна" по разбросу выборов
picks = train[['search_location_id', 'item_latitude', 'item_longitude']].join(
    train_loc_centers, on='search_location_id')
picks['dist_km'] = haversine_km(picks['lat'], picks['lon'], picks['item_latitude'], picks['item_longitude'])
loc_spread = picks.groupby('search_location_id')['dist_km'].agg(n_picks='size', dist_p50='median')
country_locs = set(
    loc_spread.index[(loc_spread['n_picks'] >= MIN_PICKS) & (loc_spread['dist_p50'] > COUNTRY_P50_KM)]
)
print('Локации без гео-фильтра:', country_locs)


def build_geo_filter(radius_km, search_centers, item_centers, no_filter_locs):
    # {search_location_id: set(item_location_id в радиусе)}, None = без фильтра
    search_centers = search_centers.dropna()
    dist = haversine_km(
        search_centers['lat'].values[:, None], search_centers['lon'].values[:, None],
        item_centers['lat'].values[None, :], item_centers['lon'].values[None, :],
    )
    item_loc_ids = item_centers.index.to_numpy()
    geo_filter = {}
    for row, loc_id in enumerate(search_centers.index):
        if loc_id in no_filter_locs:
            geo_filter[loc_id] = None
            continue
        in_radius = item_loc_ids[dist[row] <= radius_km]
        if len(in_radius) == 0:   # никого рядом - берём ближайшую
            in_radius = item_loc_ids[[dist[row].argmin()]]
        geo_filter[loc_id] = set(in_radius)
    return geo_filter


search_locs = pd.Index(pd.concat([train['search_location_id'], bq['search_location_id']]).unique())
geo_filter = build_geo_filter(R_KM, search_loc_centers.reindex(search_locs), item_loc_centers, country_locs)
print(f'Гео-фильтр: {len(geo_filter)} локаций, R = {R_KM} км')

Локации без гео-фильтра: {621540}
Гео-фильтр: 2782 локаций, R = 40 км


## Эмбеддинги

Строка в `.npy` = строка в индексном `.parquet`. items.npy ~1гб, поэтому через mmap.

In [6]:
items_index = pd.read_parquet(os.path.join(EMB_DIR, 'items_index.parquet'))
queries_index = pd.read_parquet(os.path.join(EMB_DIR, 'queries_index.parquet'))
item_emb_all = np.load(os.path.join(EMB_DIR, 'items.npy'), mmap_mode='r')
query_emb_params = np.load(os.path.join(EMB_DIR, 'queries_with_params.npy'))
query_emb_noparams = np.load(os.path.join(EMB_DIR, 'queries_no_params.npy'))

assert len(item_emb_all) == len(items_index)
assert len(query_emb_params) == len(query_emb_noparams) == len(queries_index)

# маппинги id -> номер строки эмбеддинга
item_emb_row = pd.Series(np.arange(len(items_index)), index=items_index['item_id'].to_numpy())
query_emb_row = dict(zip(
    zip(queries_index['search_query'], queries_index['search_infm_params_text'].fillna('')),
    range(len(queries_index)),
))
print('Объявления:', item_emb_all.shape, '| запросы:', query_emb_params.shape)

Объявления: (515895, 1024) | запросы: (109736, 1024)


## Фичи объявлений

Не зависят от запроса, считаю один раз:
- `rating` — рейтинг 0 это по сути "нет рейтинга", ставлю NaN, бустинг сам разберётся с пропусками. Плюс флаг `has_rating`
- `rating_smoothed` — байесовское сглаживание с весом 10, чтобы 5.0 по одному отзыву не было лучше 4.9 по 300
- `reviews_log` — log1p от кол-ва отзывов (хвост до 18к)
- цена: цена <= 1 это "договорная" → NaN + флаг `price_missing`. Абсолютная цена мало что говорит (маникюр vs ремонт квартиры), поэтому ещё `price_rel_microcat` — отклонение от медианы по микрокатегории
- `phone_hidden`, `msg_forbidden` — как есть

In [7]:
# если объявление есть и в корпусе, и в train - берём версию из корпуса
item_info = pd.concat([items, train[ITEM_COLS]]).drop_duplicates('item_id').set_index('item_id')

rating = item_info['item_rating'].where(item_info['item_rating'] > 0)
n_reviews = item_info['item_rating_reviews_count'].fillna(0)
PRIOR_WEIGHT = 10
prior_rating = rating.mean()
item_info['rating'] = rating
item_info['has_rating'] = rating.notna().astype(np.float32)
item_info['rating_smoothed'] = (
    (rating.fillna(prior_rating) * n_reviews + prior_rating * PRIOR_WEIGHT) / (n_reviews + PRIOR_WEIGHT)
)
item_info['reviews_log'] = np.log1p(n_reviews)

price = item_info['item_price'].where(item_info['item_price'] > 1)   # 1 руб = договорная
item_info['price_log'] = np.log1p(price)
item_info['price_missing'] = price.isna().astype(np.float32)
item_info['price_rel_microcat'] = (
    item_info['price_log'] - item_info.groupby('item_microcat_id')['price_log'].transform('median')
)

item_info['phone_hidden'] = item_info['item_is_phone_hidden'].astype(float)
item_info['msg_forbidden'] = item_info['item_is_message_forbidden'].astype(float)

ITEM_FEATURES = ['rating', 'has_rating', 'rating_smoothed', 'reviews_log',
                 'price_log', 'price_missing', 'price_rel_microcat',
                 'phone_hidden', 'msg_forbidden']
item_info[ITEM_FEATURES].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
rating,466797.0,4.884,0.297,1.000,4.887,5.000,5.000,5.000
has_rating,515895.0,0.905,0.293,0.000,1.000,1.000,1.000,1.000
rating_smoothed,515895.0,4.888,0.118,2.958,4.884,4.917,4.955,5.000
reviews_log,515895.0,2.874,1.604,0.000,1.792,2.890,3.970,9.814
price_log,467452.0,7.027,1.571,1.099,6.217,6.909,7.824,27.631
price_missing,515895.0,0.094,0.292,0.000,0.000,0.000,0.000,1.000
price_rel_microcat,467452.0,0.046,1.379,-9.824,-0.597,0.000,0.691,20.029
phone_hidden,515895.0,0.139,0.346,0.000,0.000,0.000,0.000,1.000
msg_forbidden,515895.0,0.024,0.153,0.000,0.000,0.000,0.000,1.000


## Кандидатогенерация

Два прохода:
- без гео — просто topk по косинусу батчами
- внутри гео-пула — группирую запросы по локации (маска одна на группу), вне пула ставлю -inf

Кандидаты = объединение. `in_geo` беру по наличию ранга в гео-списке — это ок, т.к. если объявление из пула попало в глобальный топ-50, то и внутри пула оно не ниже 50 места, а гео-список длиннее.

In [8]:
def make_corpus(item_ids):
    # id, эмбеддинги и фичи объявлений в одном порядке
    item_ids = np.asarray(item_ids)
    rows = item_emb_row.loc[item_ids].to_numpy()
    order = np.argsort(rows)                           # из mmap сильно быстрее читать по порядку
    emb = np.empty((len(rows), item_emb_all.shape[1]), dtype=np.float16)
    emb[order] = item_emb_all[rows[order]]
    return {
        'ids': item_ids,
        'emb': torch.from_numpy(emb).to(DEVICE, DTYPE),
        'locs': item_info.loc[item_ids, 'item_location_id'].to_numpy(),
        'info': item_info.loc[item_ids].reset_index(),
        'geo_masks': {},                               # кэш масок по локациям
    }


def get_geo_mask(corpus, search_location_id):
    # True = объявление проходит гео-фильтр
    cache = corpus['geo_masks']
    if search_location_id not in cache:
        pool = geo_filter.get(search_location_id)
        if pool is None:
            cache[search_location_id] = np.ones(len(corpus['ids']), dtype=bool)
        else:
            cache[search_location_id] = np.isin(corpus['locs'], np.fromiter(pool, dtype=np.int64))
    return cache[search_location_id]


def make_query_table(df):
    # строка на запрос (индекс = qi) + всё что нужно дальше
    q = df.reset_index(drop=True).copy()
    q['emb_row'] = [query_emb_row[(t, p)] for t, p in
                    zip(q['search_query'], q['search_infm_params_text'].fillna(''))]
    centers = search_loc_centers.reindex(q['search_location_id'])
    q['center_lat'] = centers['lat'].to_numpy()
    q['center_lon'] = centers['lon'].to_numpy()
    q['is_country'] = q['search_location_id'].isin(country_locs).astype(np.float32)
    q['text_norm'] = norm(q['search_query'])
    if 'query_group' not in q:            # у бенчмарка групп нет
        q['query_group'] = -1
    return q


@torch.no_grad()
def retrieve_candidates(queries, corpus, k_geo=K_GEO, k_global=K_GLOBAL, batch_size=512):
    # на выходе: qi, ci (индекс в корпусе), rank_geo, rank_global, in_geo
    assert k_global <= k_geo, 'иначе in_geo нельзя определять по rank_geo'
    Q = torch.from_numpy(query_emb_params[queries['emb_row'].to_numpy()]).to(DEVICE, DTYPE)
    C = corpus['emb']

    # без гео
    parts = []
    for start in range(0, len(Q), batch_size):
        top = torch.topk(Q[start:start + batch_size] @ C.T, k_global, dim=1).indices.cpu().numpy()
        n = len(top)
        parts.append(pd.DataFrame({'qi': np.repeat(np.arange(start, start + n), k_global),
                                   'ci': top.ravel(),
                                   'rank_global': np.tile(np.arange(1, k_global + 1), n)}))
    global_cands = pd.concat(parts, ignore_index=True)

    # внутри гео-пула, по локациям
    locs = queries['search_location_id'].to_numpy()
    parts = []
    for loc_id in tqdm(np.unique(locs), desc='Гео-поиск', leave=False):
        q_idx = np.flatnonzero(locs == loc_id)
        mask = get_geo_mask(corpus, loc_id)
        k = int(min(k_geo, mask.sum()))
        if k == 0:
            continue   # в пуле пусто - останутся только глобальные кандидаты
        outside = torch.from_numpy(~mask).to(DEVICE)
        for start in range(0, len(q_idx), batch_size):
            batch = q_idx[start:start + batch_size]
            scores = (Q[batch] @ C.T).masked_fill_(outside, float('-inf'))
            top = torch.topk(scores, k, dim=1).indices.cpu().numpy()
            parts.append(pd.DataFrame({'qi': np.repeat(batch, k),
                                       'ci': top.ravel(),
                                       'rank_geo': np.tile(np.arange(1, k + 1), len(batch))}))
    geo_cands = pd.concat(parts, ignore_index=True)

    cands = geo_cands.merge(global_cands, on=['qi', 'ci'], how='outer')
    cands['in_geo'] = cands['rank_geo'].notna().astype(np.float32)
    cands['item_id'] = corpus['ids'][cands['ci'].to_numpy()]
    return cands.sort_values(['qi', 'ci'], ignore_index=True)


@torch.no_grad()
def pairwise_cos(query_emb, emb_rows, corpus, qi, ci, chunk=200_000):
    # косинус по парам (эмбеддинги нормированы, так что это просто dot product)
    Q = torch.from_numpy(query_emb[emb_rows]).to(DEVICE, DTYPE)
    out = np.empty(len(qi), dtype=np.float32)
    for s in range(0, len(qi), chunk):
        q_t = torch.tensor(qi[s:s + chunk], device=DEVICE)
        c_t = torch.tensor(ci[s:s + chunk], device=DEVICE)
        out[s:s + chunk] = (Q[q_t] * corpus['emb'][c_t]).sum(dim=1).float().cpu().numpy()
    return out

## Фичи пары запрос-кандидат

- сходство: `cos` (запрос с фильтрами), `cos_noparams`, `cos_rank` (место в запросе), `cos_gap` (отставание от топ-1)
- гео: `dist_km`, `in_geo`, `same_location`, `is_country`
- фичи объявления из блока выше
- история "запрос → микрокатегория" из train:
  - `mc_share_query` — какая доля выборов по этому тексту запроса приходилась на микрокатегорию кандидата (NaN, если текста в train не было)
  - `query_hist_n` — сколько вообще выборов по тексту, чтобы модель понимала, насколько доверять доле
  - `mc_share_top20` — доля микрокатегории кандидата среди топ-20 по косинусу. Разметку не юзает, поэтому работает и для новых запросов (их в бенче ~63%)

In [9]:
# --- история запрос -> микрокатегория ---
# считаю только по train без валидации, иначе валидация подсмотрит свои ответы
history = train.loc[~train['query_group'].isin(val_groups), ['query_group', 'search_query', 'item_microcat_id']].copy()
history['text_norm'] = norm(history['search_query'])

text_mc_counts = history.groupby(['text_norm', 'item_microcat_id']).size().rename('text_mc_n').reset_index()
text_counts = history.groupby('text_norm').size().rename('text_n').reset_index()

# leave-one-out: для запросов из трейна бустинга вычитаю их собственный вклад,
# иначе в фиче будет сам ответ и модель начнёт ей слепо доверять (на бенче такого не будет).
# у валидации и бенчмарка вклада в историю нет, там вычитается 0
group_mc_counts = history.groupby(['query_group', 'item_microcat_id']).size().rename('own_mc_n').reset_index()
group_counts = history.groupby('query_group').size().rename('own_n').reset_index()
print(f'Текстов запросов в истории: {len(text_counts)}')


def add_history_features(c, queries):
    qi = c['qi'].to_numpy()
    c['text_norm'] = queries['text_norm'].to_numpy()[qi]
    c['query_group'] = queries['query_group'].to_numpy()[qi]

    c = (c.merge(text_mc_counts, left_on=['text_norm', 'microcat'], right_on=['text_norm', 'item_microcat_id'], how='left')
          .drop(columns='item_microcat_id')
          .merge(text_counts, on='text_norm', how='left')
          .merge(group_mc_counts, left_on=['query_group', 'microcat'], right_on=['query_group', 'item_microcat_id'], how='left')
          .drop(columns='item_microcat_id')
          .merge(group_counts, on='query_group', how='left'))
    fill = ['text_mc_n', 'text_n', 'own_mc_n', 'own_n']
    c[fill] = c[fill].fillna(0)

    total = c['text_n'] - c['own_n']
    c['query_hist_n'] = total
    c['mc_share_query'] = np.where(total > 0, (c['text_mc_n'] - c['own_mc_n']) / total.clip(lower=1), np.nan)

    # доля микрокатегории среди топ-20 по косинусу
    top20 = c[c['cos_rank'] <= 20]
    share = (top20.groupby(['qi', 'microcat']).size() / top20.groupby('qi').size()).rename('mc_share_top20')
    c = c.merge(share, left_on=['qi', 'microcat'], right_index=True, how='left')
    c['mc_share_top20'] = c['mc_share_top20'].fillna(0)

    return c.drop(columns=fill + ['text_norm', 'query_group']).sort_values(['qi', 'ci'], ignore_index=True)


SIM_FEATURES = ['cos', 'cos_noparams', 'cos_rank', 'cos_gap']
GEO_FEATURES = ['dist_km', 'in_geo', 'same_location', 'is_country']
HIST_FEATURES = ['mc_share_query', 'query_hist_n', 'mc_share_top20']
FEATURES = SIM_FEATURES + GEO_FEATURES + ITEM_FEATURES + HIST_FEATURES


def build_features(cands, queries, corpus):
    c = cands
    qi, ci = c['qi'].to_numpy(), c['ci'].to_numpy()
    emb_rows = queries['emb_row'].to_numpy()
    info = corpus['info']

    # сходство
    c['cos'] = pairwise_cos(query_emb_params, emb_rows, corpus, qi, ci)
    c['cos_noparams'] = pairwise_cos(query_emb_noparams, emb_rows, corpus, qi, ci)
    c['cos_rank'] = c.groupby('qi')['cos'].rank(ascending=False, method='first')
    c['cos_gap'] = c.groupby('qi')['cos'].transform('max') - c['cos']

    # гео
    c['dist_km'] = haversine_km(
        queries['center_lat'].to_numpy()[qi], queries['center_lon'].to_numpy()[qi],
        info['item_latitude'].to_numpy()[ci], info['item_longitude'].to_numpy()[ci],
    )
    c['same_location'] = (queries['search_location_id'].to_numpy()[qi]
                          == info['item_location_id'].to_numpy()[ci]).astype(np.float32)
    c['is_country'] = queries['is_country'].to_numpy()[qi]

    # объявление
    for col in ITEM_FEATURES:
        c[col] = info[col].to_numpy()[ci]

    # история
    c['microcat'] = info['item_microcat_id'].to_numpy()[ci]
    c = add_history_features(c, queries)

    c[FEATURES] = c[FEATURES].astype(np.float32)
    return c

Текстов запросов в истории: 72924


## Отбор топ-50 и метрика

`select_top`: сначала `TOP_N - n_global` лучших из гео-пула, потом добиваю до 50 лучшими из оставшихся. При равных скорах выше тот, у кого косинус больше.

С `score_col='cos'` это ровно мой старый бейзлайн, так что модель и бейзлайн сравниваются на одних и тех же кандидатах.

In [10]:
def select_top(cands, score_col, n_total=TOP_N, n_global=N_GLOBAL):
    # {qi: [item_id, ...]}, максимум n_total на запрос
    sort_cols = [score_col] if score_col == 'cos' else [score_col, 'cos']
    df = cands[['qi', 'item_id', 'in_geo'] + sort_cols].sort_values(
        ['qi'] + sort_cols, ascending=[True] + [False] * len(sort_cols), kind='stable')

    geo = df[df['in_geo'] == 1]
    first = geo[geo.groupby('qi').cumcount() < n_total - n_global]           # лучшие из гео-пула

    rest = df.drop(first.index)
    need = n_total - first.groupby('qi').size().reindex(df['qi'].unique(), fill_value=0)
    second = rest[rest.groupby('qi').cumcount() < rest['qi'].map(need)]      # добиваем до 50

    return pd.concat([first, second]).groupby('qi')['item_id'].apply(list).to_dict()


def recall_per_query(selected, answers):
    return np.array([len(set(selected.get(qi, [])) & true) / len(true) for qi, true in enumerate(answers)])


def candidates_ceiling(cands, answers):
    # потолок = recall, если бы брали всех кандидатов
    return recall_per_query(cands.groupby('qi')['item_id'].apply(set).to_dict(), answers).mean()

## Датасеты

Корпус один на всех: объявления бенчмарка + выбранные объявления запросов fit/es/val (иначе правильных ответов просто нет в корпусе). Заодно ответы одних запросов становятся негативами для других — иначе все подмешанные из train объявления были бы позитивами, и модель могла бы выучить "объявление из train", а не то что нужно.

Таргет — выбрано/нет, вес позитива 1/кол-во ответов. Из fit выкидываю запросы без позитивов в кандидатах (учить нечему), в es/val оставляю всё, чтобы recall был честный.

In [11]:
def make_query_set(groups):
    # таблица запросов + список ответов в том же порядке
    rows = train[train['query_group'].isin(groups)]
    queries = make_query_table(rows.drop_duplicates('query_group').sort_values('query_group'))
    answers = rows.groupby('query_group')['item_id'].apply(set).loc[queries['query_group']].tolist()
    return queries, answers


def add_labels_and_weights(cands, answers):
    pos = pd.DataFrame([(qi, item) for qi, s in enumerate(answers) for item in s], columns=['qi', 'item_id'])
    pos['label'] = 1
    cands = cands.merge(pos, on=['qi', 'item_id'], how='left')
    cands['label'] = cands['label'].fillna(0).astype(np.int32)
    n_answers = np.array([len(a) for a in answers])
    cands['weight'] = np.where(cands['label'] == 1, 1 / n_answers[cands['qi'].to_numpy()], 1.0)
    return cands


query_sets = {
    'fit': make_query_set(fit_groups),
    'es': make_query_set(es_groups),
    'val': make_query_set(val_groups),
}

extra_ids = set().union(*[a for _, answers in query_sets.values() for a in answers]) - set(items['item_id'])
search_corpus = make_corpus(np.concatenate([items['item_id'].to_numpy(), np.array(sorted(extra_ids))]))
print(f'Корпус: {len(search_corpus["ids"])} объявлений (добавлено из train: {len(extra_ids)})')

datasets = {}
for name, (queries, answers) in query_sets.items():
    cands = retrieve_candidates(queries, search_corpus)
    cands = build_features(cands, queries, search_corpus)
    cands = add_labels_and_weights(cands, answers)
    datasets[name] = cands
    print(f'{name:4s}: запросов {len(queries)}, кандидатов {len(cands)} '
          f'({len(cands) / len(queries):.0f} на запрос), потолок recall {candidates_ceiling(cands, answers):.4f}')

has_pos = datasets['fit'].groupby('qi')['label'].transform('max') == 1
datasets['fit'] = datasets['fit'][has_pos].reset_index(drop=True)
print(f'fit после фильтра: {datasets["fit"]["qi"].nunique()} запросов, '
      f'доля позитивов {datasets["fit"]["label"].mean():.4f}')

Корпус: 204087 объявлений (добавлено из train: 14875)


Гео-поиск:   0%|          | 0/921 [00:00<?, ?it/s]

fit : запросов 9000, кандидатов 1301691 (145 на запрос), потолок recall 0.9055


Гео-поиск:   0%|          | 0/326 [00:00<?, ?it/s]

es  : запросов 1000, кандидатов 145018 (145 на запрос), потолок recall 0.9092


Гео-поиск:   0%|          | 0/468 [00:00<?, ?it/s]

val : запросов 2500, кандидатов 359016 (144 на запрос), потолок recall 0.8861
fit после фильтра: 8230 запросов, доля позитивов 0.0091


## Обучение

LGBMClassifier, early stopping по logloss на es. Сначала пробовал lambdarank (LGBMRanker), но на cpu он учился очень долго из-за попарных сравнений внутри запроса, а порядок внутри топ-50 на метрику не влияет — так что бинарки хватает. `deterministic=True` + сид для воспроизводимости.

In [12]:
class TqdmCallback:
    # прогрессбар по деревьям, чтобы видеть что обучение живое
    def __init__(self, total):
        self.bar = tqdm(total=total, desc='Обучение', unit='дерево')
        self.best = np.inf

    def __call__(self, env):
        self.bar.update(1)
        if env.evaluation_result_list:
            _, metric, value, _ = env.evaluation_result_list[0]
            self.best = min(self.best, value)
            self.bar.set_postfix({metric: f'{value:.5f}', 'лучшая': f'{self.best:.5f}'})


model = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    deterministic=True, force_row_wise=True,
    verbose=-1,
)

fit_df, es_df = datasets['fit'], datasets['es']
progress = TqdmCallback(total=model.n_estimators)
model.fit(
    fit_df[FEATURES], fit_df['label'], sample_weight=fit_df['weight'],
    eval_set=[(es_df[FEATURES], es_df['label'])], eval_sample_weight=[es_df['weight']],
    eval_metric='binary_logloss',
    callbacks=[progress, lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
)
progress.bar.close()
print('Лучшее число деревьев:', model.best_iteration_)
model.booster_.save_model(os.path.join(OUT_DIR, 'lgbm_binary.txt'))


def predict_score(df):
    return model.predict_proba(df[FEATURES])[:, 1]


importance = pd.Series(model.booster_.feature_importance('gain'), index=FEATURES)
(importance / importance.sum()).sort_values(ascending=False).round(4)

Обучение:   0%|          | 0/2000 [00:00<?, ?дерево/s]

Лучшее число деревьев: 290


,0
dist_km,0.2497
cos_rank,0.1618
cos_noparams,0.1112
mc_share_top20,0.1048
same_location,0.0953
mc_share_query,0.0602
cos,0.0475
cos_gap,0.0412
in_geo,0.0336
query_hist_n,0.0235


## Скор

recall@50 на es и val на одних и тех же кандидатах: потолок / отбор по косинусу / отбор моделью.

In [13]:
rows = []
for name in ['es', 'val']:
    df, answers = datasets[name], query_sets[name][1]
    df['score'] = predict_score(df)
    rows.append({'выборка': name, 'вариант': 'потолок', 'recall@50': candidates_ceiling(df, answers)})
    for n_global in [0, 3, 5]:
        for score_col, label in [('cos', 'косинус'), ('score', 'модель')]:
            recall = recall_per_query(select_top(df, score_col, n_global=n_global), answers).mean()
            rows.append({'выборка': name, 'вариант': f'{label}, N_GLOBAL={n_global}', 'recall@50': recall})

pd.DataFrame(rows).pivot(index='вариант', columns='выборка', values='recall@50').round(4)

выборка,es,val
вариант,,
"косинус, N_GLOBAL=0",0.8561,0.8151
"косинус, N_GLOBAL=3",0.8602,0.8184
"косинус, N_GLOBAL=5",0.8584,0.8172
"модель, N_GLOBAL=0",0.8775,0.8421
"модель, N_GLOBAL=3",0.8879,0.8568
"модель, N_GLOBAL=5",0.8879,0.8580
потолок,0.9092,0.8861


Модель на валидации даёт ~0.857 против 0.818 у чистого косинуса, до потолка кандидатов (0.886) остаётся ~3 п.п. N_GLOBAL=3 и 5 почти одинаковые, оставил 3.

Посмотрим по сегментам, где модель помогает, а где нет:

In [14]:
val_q, val_answers = query_sets['val']
val_df = datasets['val']
known_texts = set(norm(train[~train['query_group'].isin(val_groups)]['search_query']))

seg = pd.DataFrame({
    'знакомый текст': norm(val_q['search_query']).isin(known_texts).to_numpy(),
    'вся страна': (val_q['is_country'] == 1).to_numpy(),
    'правильных > 1': [len(a) > 1 for a in val_answers],
    'косинус': recall_per_query(select_top(val_df, 'cos'), val_answers),
    'модель': recall_per_query(select_top(val_df, 'score'), val_answers),
})
for col in ['знакомый текст', 'вся страна', 'правильных > 1']:
    print(seg.groupby(col)[['косинус', 'модель']].agg(['mean', 'size']).round(4), '\n')

               косинус        модель      
                  mean  size    mean  size
знакомый текст                            
False           0.7977  1575  0.8386  1575
True            0.8536   925  0.8877   925 

           косинус        модель      
              mean  size    mean  size
вся страна                            
False        0.827  2430  0.8673  2430
True         0.519    70  0.4905    70 

               косинус        модель      
                  mean  size    mean  size
правильных > 1                            
False           0.8129  2304  0.8524  2304
True            0.8822   196  0.9080   196 



Выводы по сегментам:
- модель бустит и знакомые (+3.4 п.п.), и новые тексты (+4 п.п.), т.е. работает не только за счёт истории из train
- самое слабое место — поиск по "всей России" (621540): recall ~0.5, и модель там даже чуть хуже косинуса. Там нет гео, и кандидаты берутся из всего корпуса, так что среди похожих объявлений угадать нужное сложно. Таких запросов в бенче ~4%, это кандидат на отдельную доработку
- у запросов с несколькими ответами recall выше (проще попасть хотя бы в часть)

## Сабмит

Тот же пайплайн, только корпус — чисто `benchmark_items`.

In [15]:
N_GLOBAL_FINAL = N_GLOBAL

bench_corpus = make_corpus(items['item_id'].to_numpy())
bench_q = make_query_table(bq)
bench_df = retrieve_candidates(bench_q, bench_corpus)
bench_df = build_features(bench_df, bench_q, bench_corpus)
bench_df['score'] = predict_score(bench_df)

selected = select_top(bench_df, 'score', n_global=N_GLOBAL_FINAL)
answer = pd.DataFrame({
    'query_id': bench_q['query_id'].to_numpy(),
    'answer': [' '.join(selected[qi]) for qi in range(len(bench_q))],
})
answer.head()

Гео-поиск:   0%|          | 0/273 [00:00<?, ?it/s]

,query_id,answer
0,70DfDUpwjxB4lzFd,6042fecdd7753112 a1df0525799ab4b5 d722bcda1a55...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 cbeccbecb1fb8d86 1c6d2079e07f...
2,LZCZNoVG4AFUkVRJ,d8b9d06b50158c04 d8fce513e4f000a7 3b370cc603f6...
3,660ac9QVtXkRxZC3,3cce43c811459438 a846a2a5241e1180 e42795fcb729...
4,YgHcM9MVbxKnxD1e,1a62634394544781 53f9c8caeeb8111e 13f58fe9542b...


In [16]:
def check_answer(answer_df):
    # проверки из условия, чтобы не слить попытку на формате
    valid_ids = set(items['item_id'])
    assert list(answer_df.columns) == ['query_id', 'answer'], 'колонки должны быть ровно query_id и answer'
    assert answer_df['query_id'].is_unique, 'есть повторяющиеся query_id'
    assert set(answer_df['query_id']) == set(bq['query_id']), 'набор query_id не совпадает с бенчмарком'
    for ans in answer_df['answer']:
        ids = ans.split(' ')
        assert len(ids) <= TOP_N, 'больше 50 объявлений в строке'
        assert len(ids) == len(set(ids)), 'повторы внутри строки'
        assert all(i in valid_ids for i in ids), 'есть item_id не из корпуса'
    print('Формат в порядке. Средняя длина ответа:', answer_df['answer'].str.split().str.len().mean())


check_answer(answer)

answer_path = os.path.join(OUT_DIR, 'answer_boosting.csv')
answer.to_csv(answer_path, index=False)
with open(answer_path, 'rb') as f:
    print('MD5 ответа:', hashlib.md5(f.read()).hexdigest())   # сверка с отправленным файлом

try:
    from google.colab import files
    files.download(answer_path)
except ImportError:
    pass

Формат в порядке. Средняя длина ответа: 50.0
MD5 ответа: c34aa37bace57ca886bd68fff0cd8683


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>